In [1]:
import pandas as pd 
import re

# Read the data from the CSV file (using absolute path)
df = pd.read_csv('/home/sheikh/Projects/Thesis/data/medium_clean.csv')
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nFirst 3 rows:\n", df.head(3))

Shape: (1165, 4)

Dtypes:
 grmd               int64
md_name           object
tex_text          object
tex_text_clean    object
dtype: object

First 3 rows:
    grmd                    md_name  \
0  1436   LIMNOCHORDIA L945 MEDIUM   
1  1479   SOLIDESULFOVIBRIO MEDIUM   
2  1481  Bold's Basal Medium (BBM)   

                                            tex_text  \
0  \mono{KH$_2$PO$_4$} {0.2} {g}\n\mono{MgCl$_2$$...   
1  \mono{KH$_2$PO$_4$}                           ...   
2  \mono{Agar}{20g}\\mono{ Distilled water}{980mL...   

                                      tex_text_clean  
0  \mono{KH$_2$PO$_4$} {0.2} {g}\n\mono{MgCl$_2$·...  
1  \mono{KH$_2$PO$_4$} {0.20}{g}\n\mono{NH$_4$Cl}...  
2  \mono{Agar} {20} {g}\n\mono{Distilled water} {...  


In [2]:
RE_TAG = re.compile(r"\s*\\(mono|chu)")

def read_chu_block(lines, i):
    r"""Accumulate a \chu block starting at line i.

    Returns (text, next_index). Terminates at whichever comes first:
      - the brace that closes an opened block
      - the next line starting with \mono or \chu
      - end of text

    Two source quirks make the brace unreliable on its own: 28 mediums have
    \chu blocks with no closing brace, and bare tags like \chuJCM and
    \chuSolutionB open no brace at all.
    """
    buf = []
    depth = 0
    opened = False

    for n in range(i, len(lines)):
        line = lines[n]

        if n > i and RE_TAG.match(line):
            return " ".join(buf).strip(), n

        buf.append(line.strip())
        depth += line.count("{") - line.count("}")

        if depth > 0:
            opened = True
        elif opened:
            return " ".join(buf).strip(), n + 1

    return " ".join(buf).strip(), len(lines)

In [3]:
# Demonstrate the parser on a few sample records to inspect the extracted blocks.
for grmd in [1205, 50, 1326]:
    # Retrieve the LaTeX text for the selected record and split it into lines.
    lines = df[df["grmd"] == grmd]["tex_text_clean"].values[0].splitlines()
    i = 0
    print(f"\n=== {grmd} ===")
    while i < len(lines):
        # Only parse blocks that begin with a \chu tag.
        if RE_TAG.match(lines[i]) and "\\chu" in lines[i]:
            text, i = read_chu_block(lines, i)
            print(f"  BLOCK: {text[:100]!r}")
        else:
            i += 1


=== 1205 ===
  BLOCK: '\\chu{Mix components thoroughly, adjust pH to 7.0 and autoclave under a N$_2$ gas atmosphere.  After '
  BLOCK: '\\chu{Distribute the medium into culture vessels (e.g., 20 ml in Balch tubes) and seal with butyl rub'
  BLOCK: '\\chu{10mM Fe quinate solution:}'
  BLOCK: '\\chu{Comment:}'
  BLOCK: '\\chuJCM 32405 may grow better in a modified medium by the addition of 20 mM sodium lactate and 50 μM'

=== 50 ===
  BLOCK: '\\chu{Cook or steam 20.0 g of oatmeal in 1.0 L of distilled water for 20 min. Filter through cheesecl'
  BLOCK: '\\chu{Adjust pH to 7.2.}'
  BLOCK: '\\chu{Trace salts solution:}'

=== 1326 ===
  BLOCK: '\\chu{Solution A:}'
  BLOCK: '\\chuSolutionB:}'
  BLOCK: '\\chuSolutionC:}'
  BLOCK: '\\chu{Dissolve ingredients of each solution in the appropriate amount of distilled water. Bring solut'


In [4]:
# blocks that look suspiciously long — a block that swallowed the ones after it
lengths = []
for _, row in df.iterrows():
    lines = row["tex_text_clean"].splitlines()
    i = 0
    while i < len(lines):
        if RE_TAG.match(lines[i]) and "\\chu" in lines[i]:
            text, i = read_chu_block(lines, i)
            lengths.append((len(text), row["grmd"], text[:80]))
        else:
            i += 1

lengths.sort(reverse=True)
for n, g, t in lengths[:10]:
    print(f"{n:5}  {g}  {t!r}")

  912  355  '\\chu{Mix ingredients, except methanol, L--cysteine·HCl·H$_2$O and Na$_2$S·9H$_2$'
  893  1148  '\\chu{Comments: Typically, prepare the medium in glass bottles with plastic caps '
  857  236  "\\chu{Combine Modified Brock's salt base solution, trisodium citrate and resazuri"
  850  462  '\\chu{Dissolve components except NaHCO$_3$, Vitamin solution, Thiamine solution, '
  849  484  '\\chu{Mix ingredients except Na$_2$S$_2$O$_3$·5H$_2$O, yeast extract, Trace vitam'
  843  628  '\\chu{Autoclave both layer solutions.  Shortly after autoclaving, pipette the bot'
  833  378  '\\chu{Dissolve ingredients, except crotonic acid, NaHCO$_3$, L--cysteine·HCl·H$_2'
  817  243  '\\chu{Mix ingredients, except Trace vitamins, KHCO$_3$, L--cysteine·HCl·H$_2$O an'
  764  266  '\\chu{Mix ingredients, except NaHCO$_3$, L--cysteine·HCl·H$_2$O and Na$_2$S·9H$_2'
  762  251  '\\chu{Mix ingredients, except NaHCO$_3$, L--cysteine·HCl·H$_2$O and Na$_2$S·9H$_2'


In [5]:
def split_mono(line):
    """Extract (name, amount, unit) from a \\mono line, respecting nesting.

    Returns None if the line cannot be read.
    """
    # Locate the opening \mono tag.
    i = line.find("\\mono")
    if i == -1:
        return None

    # Find the first argument block that follows the tag.
    j = line.find("{", i)
    if j == -1:
        return None

    # Walk through the braces to find the matching closing brace.
    depth = 0
    for k in range(j, len(line)):
        if line[k] == "{":
            depth += 1
        elif line[k] == "}":
            depth -= 1
            if depth == 0:
                name, rest = line[j + 1:k], line[k + 1:]
                break
    else:
        return None

    # Extract the two value fields inside the remaining braces.
    vals = re.findall(r"\{([^{}]*)\}", rest)
    if len(vals) < 2:
        return None
    return name.strip(), vals[0].strip(), vals[1].strip()

In [6]:
from collections import Counter
tag_kinds = Counter()
errors = []

# Scan every row, looking for \mono lines and \chu blocks.
for _, row in df.iterrows():
    lines = row["tex_text_clean"].splitlines()
    i = 0
    try:
        while i < len(lines):
            line = lines[i]
            if "\\mono" in line:
                # Record malformed \mono entries so they can be inspected later.
                if split_mono(line) is None:
                    errors.append((row["grmd"], "mono", line[:60]))
                i += 1
            elif RE_TAG.match(line) and "\\chu" in line:
                # Parse the full block and count how many were successfully read.
                text, i = read_chu_block(lines, i)
                tag_kinds["chu_block"] += 1
            else:
                i += 1
    except Exception as e:
        # Keep track of any unexpected parsing exceptions.
        errors.append((row["grmd"], type(e).__name__, str(e)[:60]))

print("blocks read:", tag_kinds["chu_block"])
print("errors:", len(errors))
for e in errors[:10]:
    print(" ", e)

blocks read: 2313
errors: 0


In [7]:
# Count repeated headings extracted from \chu blocks for quick inspection.
heads = Counter()
for _, row in df.iterrows():
    lines = row["tex_text_clean"].splitlines()
    i = 0
    while i < len(lines):
        if RE_TAG.match(lines[i]) and "\\chu" in lines[i]:
            text, i = read_chu_block(lines, i)
            heads[text[:45]] += 1
        else:
            i += 1

for h, c in heads.most_common(40):
    print(f"{c:4}  {h!r}")

  59  '\\chu{Solution A:}'
  53  '\\chu{Add components to distilled water and br'
  51  '\\chu{Adjust pH to 7.0.}'
  47  '\\chu{Mix components thoroughly and adjust pH '
  47  '\\chu{Solution B:}'
  44  '\\chu{Mix components thoroughly and autoclave '
  40  '\\chu{Mix components thoroughly, adjust pH to '
  35  '\\chu{Vitamin solution:}'
  30  '\\chu{Trace element solution:}'
  24  '\\chu{Adjust pH to 7.2.}'
  22  '\\chu{Trace mineral solution:}'
  19  '\\chu{Adjust pH to 6.8.}'
  18  '\\chu{Mix components thoroughly, bring to a bo'
  17  '\\chu{Adjust pH to 7.5.}'
  17  '\\chu{Distribute the medium into culture vesse'
  17  '\\chu{Dissolve nitrilotriacetic acid and adjus'
  17  '\\chu{Solution C:}'
  16  '\\chu{Mix components and autoclave under a N$_'
  14  '\\chu{Aseptically distribute the medium into c'
  11  '\\chu{Aseptically and anaerobically distribute'
  10  '\\chu{Adjust pH to 7.3.}'
  10  '\\chu{For preparation of solid medium, add 20.'
   9  '\\chu{To complete the medium, a

In [8]:
clean = df.copy()

In [9]:
for _, row in clean.iterrows():
    lines = row["tex_text_clean"].splitlines()
    i = 0
    while i < len(lines):
        if RE_TAG.match(lines[i]) and "\\chu" in lines[i]:
            text, j = read_chu_block(lines, i)
            if text.strip() in ("\\chu{}", "\\chu{"):
                print(row["grmd"], "→", [l.strip()[:60] for l in lines[max(0,i-1):i+3]])
            i = j
        else:
            i += 1

1303 → ['\\chu{Mix components, adjust pH to 6.0-6.2. After autoclaving', '\\chu{}', '', '\\mono{Bovine calf serum (heat-inactivated)} {5.0}{ml}']
1303 → ['\\mono{1% Urea - 4% yeast extract solution (see below)} {10.0', '\\chu{}', '', '\\chu{1% Phenol red solution:}']
1303 → ['\\chu{Dissolve 1.0 g phenol red first in 10-20 ml of 0.1 N Na', '\\chu{}', '', '\\chu{1% Urea - 4% yeast extract solution:}']
1303 → ['\\chu{Mix components, sterilize by filtration through a 0.22 ', '\\chu{}', '', '\\chu{Dispense the medium into sterilized plastic tubes (e.g.']
1170 → ['', '\\chu{}', '\\chu{Mineral solution:}', '']
1004 → ['1 ml of this solution is added to 100 ml of the medium (cont', '\\chu{}', '\\chu{[Note]}', '\\chu{Adjust pH to 7.0 for JCM 31640.}']
685 → ['', '\\chu{}', '', '\\chu{Solution B:}']
685 → ['', '\\chu{}', '', '\\chu{To complete the medium, add 70 ml Solution B to 1.0 L S']


In [10]:
tex = clean[clean["grmd"] == 1004]["tex_text_clean"].values[0]
lines = tex.splitlines()
for n, l in enumerate(lines):
    if "[Note]" in l:
        for x in lines[max(0, n-3): n+4]:
            print(repr(x.strip()))
        break

'\\chu{Dissolve NaOH in 300 ml distilled water.  While stirring vigorously, slowly add HEPES until completely dissolved.  Fill up to 450 ml with distilled water.  If'
'1 ml of this solution is added to 100 ml of the medium (containing all additive but HEPES), pH should be around 7.6 at 30ºC.  Otherwise, adjust pH of the HEPES solution with 10 N NaOH or conc. HCl.  Finally fill up to 500 ml with distilled water.}'
'\\chu{}'
'\\chu{[Note]}'
'\\chu{Adjust pH to 7.0 for JCM 31640.}'
''


In [11]:
empty = 0
for _, row in clean.iterrows():
    lines = row["tex_text_clean"].splitlines()
    i = 0
    while i < len(lines):
        if RE_TAG.match(lines[i]) and "\\chu" in lines[i]:
            text, i = read_chu_block(lines, i)
            inner = re.search(r"\\chu\w*\{(.*)\}", text, re.S)
            if inner is not None and not inner.group(1).strip():
                empty += 1
        else:
            i += 1
print("empty blocks:", empty)

empty blocks: 8


In [12]:
for _, row in clean.iterrows():
    lines = row["tex_text_clean"].splitlines()
    i = 0
    while i < len(lines):
        if RE_TAG.match(lines[i]) and "\\chu" in lines[i]:
            text, i = read_chu_block(lines, i)
            inner = re.search(r"\\chu\w*\{(.*)\}", text, re.S)
            if inner is not None and not inner.group(1).strip():
                print(row["grmd"], "→", repr(text[:60]))
        else:
            i += 1

1303 → '\\chu{}'
1303 → '\\chu{}'
1303 → '\\chu{}'
1303 → '\\chu{}'
1170 → '\\chu{}'
1004 → '\\chu{}'
685 → '\\chu{}'
685 → '\\chu{}'


In [13]:
import re

for grmd in [1205, 1326, 1353]:
    tex = clean[clean["grmd"] == grmd]["tex_text_clean"].values[0]
    heads = re.findall(r"\\chu\w*\{?([^}]*:)\}?", tex)
    monos = [m for m in re.findall(r"\\mono\{([^}]*)\}", tex)]
    print(f"\n=== {grmd} ===")
    for h in heads:
        h = h.strip()
        ref = any(h.rstrip(':').lower() in m.lower() for m in monos)
        print(f"  {h!r:45} referenced by a \\mono: {ref}")


=== 1205 ===
  'Mix components thoroughly, adjust pH to 7.0 and autoclave under a N$_2$ gas atmosphere.  After cooling, aseptically and anaerobically add the following solutions from anaerobic stocks (autoclaved or *filter-sterilized):' referenced by a \mono: False
  '10mM Fe quinate solution:'                   referenced by a \mono: False
  'Comment:'                                    referenced by a \mono: False

=== 1326 ===
  'Solution A:'                                 referenced by a \mono: False
  ':'                                           referenced by a \mono: True
  ':'                                           referenced by a \mono: True

=== 1353 ===
  'Mix components thoroughly and adjust pH to 7.5 with NaOH.  Distribute the medium into culture vessels (e.g., 5.0 ml in 25 ml serum bottles/Balch tubes) under a stream of N$_2$, seal with butyl rubber stoppers and autoclave.  After cooling, aseptically add per liter the following solutions (autoclaved or filter-sterili

In [ ]:
def norm(s):
    """Normalize a string for robust comparison.

    This lowercases the input and removes any character that is not
    a lowercase ASCII letter or digit. It's used to compare headings and
    ingredient names while ignoring punctuation, spacing, and case.

    Example: "Solution A" -> "solutiona"
    """
    return re.sub(r"[^a-z0-9]", "", s.lower())

for grmd in [1205, 1353]:
    tex = clean[clean["grmd"] == grmd]["tex_text_clean"].values[0]
    heads = [h.strip() for h in re.findall(r"\\chu\{([^}]*:)\}", tex)]
    monos = re.findall(r"\\mono\{([^}]*)\}", tex)
    print(f"\n=== {grmd} ===")
    for h in heads:
        key = norm(h.rstrip(":"))
        ref = any(key in norm(m) for m in monos)
        print(f"  {h[:50]!r:55} referenced: {ref}")


=== 1205 ===
  'Mix components thoroughly, adjust pH to 7.0 and au'    referenced: False
  '10mM Fe quinate solution:'                             referenced: True
  'Comment:'                                              referenced: False

=== 1353 ===
  'Mix components thoroughly and adjust pH to 7.5 wit'    referenced: False
  'Mineral solution A:'                                   referenced: True
  'Trace vitamins solution:'                              referenced: True
  'Mineral solution B:'                                   referenced: True


In [ ]:
def norm(s):
    """Normalize a string for comparison across headings and ingredients.

    Converts `s` to lowercase and removes non-alphanumeric characters
    so that comparisons ignore spacing, punctuation, and case.
    """
    return re.sub(r"[^a-z0-9]", "", s.lower())

for _, row in clean.iterrows():
    tex = row["tex_text_clean"]
    heads = [h.strip().rstrip(":") for h in re.findall(r"\\chu\{([^}]*:)\}", tex)]
    monos = re.findall(r"\\mono\{([^}]*)\}", tex)
    for h in heads:
        key = norm(h)
        if not key:
            continue
        hits = [m for m in monos if key in norm(m)]
        if hits and not any("see below" in m for m in hits):
            print(f"{row['grmd']}  heading {h[:40]!r}  →  matched WITHOUT 'see below': {hits[0][:60]!r}")

1347  heading 'Trace minerals'  →  matched WITHOUT 'see below': 'Trace minerals (see Medium No. [151])'
1345  heading 'Trace vitamins'  →  matched WITHOUT 'see below': 'Trace vitamins (see Medium No. [197])'
1345  heading 'FeCl$_2$ solution'  →  matched WITHOUT 'see below': 'FeCl$_2$ solution (see Medium No. [187])'
1345  heading 'Trace element solution'  →  matched WITHOUT 'see below': 'Trace element solution (see Medium No. [187])'
1294  heading 'Modified MJ synthetic seawater C'  →  matched WITHOUT 'see below': 'Modified MJ synthetic seawater C'
1199  heading 'Phosphates solution'  →  matched WITHOUT 'see below': 'Phosphates solution'
815  heading 'Phosphate buffer stock solution'  →  matched WITHOUT 'see below': 'Phosphate buffer stock solution'
776  heading 'Trace metal solution'  →  matched WITHOUT 'see below': 'Trace metal solution'
547  heading 'Solution A'  →  matched WITHOUT 'see below': '5\\% K$_2$HPO$_4$ solution (autoclaved)'


In [16]:
tex = clean[clean["grmd"] == 1345]["tex_text_clean"].values[0]
lines = tex.splitlines()
for n, l in enumerate(lines):
    if "Trace vitamins:" in l:
        for x in lines[n:n+6]:
            print(repr(x.strip()[:80]))
        break

'\\chu{Trace vitamins:}'
'\\mono{Biotin} {2.0}{mg}'
'\\mono{Folic acid} {2.0}{mg}'
'\\mono{Pyridoxine·HCl} {10.0}{mg}'
'\\mono{Thiamine·HCl} {5.0}{mg}'
'\\mono{Riboflavin} {5.0}{mg}'


In [17]:
for grmd in [1347, 1294, 776]:
    tex = clean[clean["grmd"] == grmd]["tex_text_clean"].values[0]
    lines = tex.splitlines()
    for n, l in enumerate(lines):
        if re.match(r"\s*\\chu\{[^}]{1,50}:\}", l):
            nxt = lines[n+1].strip()[:60] if n+1 < len(lines) else ""
            print(f"{grmd}  {l.strip()[:45]!r}  →  {nxt!r}")

1347  "\\chu{Modified Wolfe's Mineral solution:}"  →  '\\mono{NH$_4$Cl} {1.0}{g}'
1347  '\\chu{Trace minerals:}'  →  '\\mono{Nitrilotriacetic acid} {1.5}{g}'
1347  '\\chu{Trace vitamins:}'  →  '\\mono{Biotin} {2.0}{mg}'
1294  '\\chu{Modified MJ synthetic seawater C:}'  →  '\\mono{NaCl} {25.0}{g}'
1294  '\\chu{Trace mineral solution:}'  →  '\\mono{Trace minerals (see Medium No. [151])} {1.0}{L}'
776  '\\chu{Trace metal solution:}'  →  '\\mono{CoCl$_2$·6H$_2$O} {0.3}{g}'


In [18]:
bad = []
for _, row in clean.iterrows():
    lines = row["tex_text_clean"].splitlines()
    for n, l in enumerate(lines):
        m = re.match(r"\s*\\chu\{([^}]{1,50}:)\}\s*$", l)
        if m:
            nxt = lines[n+1].strip() if n+1 < len(lines) else ""
            if not nxt.startswith("\\mono"):
                bad.append((row["grmd"], m.group(1), nxt[:50]))

print(f"headings NOT followed by an ingredient: {len(bad)}")
for b in bad[:15]:
    print(" ", b)

headings NOT followed by an ingredient: 60
  (1462, 'Ni-Se-W solution:', '')
  (1389, 'Culture supernatant:', '\\chu{Centrifuge a grown culture of strain Acc8 (=J')
  (1274, 'Comment:', '\\chu{Colloidal chitin solution can be replaced by ')
  (1419, 'Modified A5 solution:', '')
  (1408, 'Syringate solution:', '\\chu{Dissolve 6 g syringic acid in 70 ml distilled')
  (1363, 'Vitamin mixture solution:', '')
  (1342, 'Ferrihydrite slurry:', '\\chu{Slowly titrate 400 ml of 20 mM FeCl$_3$·6H$_2')
  (1303, '1% Phenol red solution:', '\\chu{Dissolve 1.0 g phenol red first in 10-20 ml o')
  (1313, 'Menadione solution:', '\\chu{Autoclave 100 mg menadione (vitamin K$_3$) wi')
  (1286, 'Sterilizationofsulfur:', '\\chu{Steam sulfur for 3 hr, or autoclave at 105C f')
  (1287, 'Sterilization of sulfur:', '\\chu{Steam sulfur for 3 hr, or autoclave at 105C f')
  (1288, 'Sterilization of sulfur:', '\\chu{Steam sulfur for 3 hr, or autoclave at 105C f')
  (1284, 'Sterilization of sulfur:', '\\chu{Steam su

In [19]:
# sub-recipe headings followed by prose rather than \mono
prose_recipes = [b for b in bad if b[2].startswith("\\chu") and "omment" not in b[1]]
print(len(prose_recipes))

36


In [20]:
for grmd in [1199, 815]:
    tex = clean[clean["grmd"] == grmd]["tex_text_clean"].values[0]
    print(f"\n=== {grmd} ===")
    print(tex[:900])


=== 1199 ===
\mono{(NH$_4$)$_2$SO$_4$} {0.50}{g}
\mono{MgSO$_4$·7H$_2$O} {0.20}{g}
\mono{CaCl$_2$·2H$_2$O} {0.05}{g}
\mono{SL-4 trace element solution (see Medium No. [340])} {10.0}{ml}
\mono{Agar} {15.0}{g}
\mono{Distilled water} {900.0}{ml}
\chu{Adjust pH to 6.9. After autoclaving, aseptically add the phosphates solution to the medium. After inoculation, put several drops of toluene inside of lid of petri dishes.}

\mono{Phosphates solution} {100.0}{ml}
\chu{Phosphates solution:}
\mono{Na$_2$HPO$_4$·2H$_2$O} {2.44}{g}
\mono{KH$_2$PO$_4$} {1.52}{g}
\mono{Distilled water} {1.0}{L}
\chu{Adjust pH to 6.9.}
     


=== 815 ===
\mono{NH$_4$Cl} {0.1}{g}
\mono{MgSO$_4$·7H$_2$O} {0.1}{g}
\mono{CaCl$_2$·2H$_2$O} {0.02}{g}
\mono{Trace element solution (see below)} {0.1}{ml}
\mono{Iron stock solution (see below)} {0.1}{ml}
\mono{Distilled water} {1.0}{L}
\chu{Mix components and autoclave. After cooling, add the following solution
(autoclaved) to the medium:}

\mono{Phosphate buffer stock soluti

In [ ]:
def norm(s):
    """Return a compact normalized form of `s` for quick comparisons.

    - Lowercases the input
    - Removes any character that is not a-z or 0-9

    Useful for comparing labels where formatting differs but words match.
    """
    return re.sub(r"[^a-z0-9]", "", s.lower())

print(repr(norm("Solution A")))
print(repr(norm("5\\% K$_2$HPO$_4$ solution (autoclaved)")))

'solutiona'
'5k2hpo4solutionautoclaved'


In [22]:
def base_name(s):
    """Strip reference markers and formatting, for comparing a heading
    against an ingredient name."""
    s = re.sub(r"\(see [^)]*\)", "", s)   # drop (see below) / (see Medium No. [X])
    s = s.replace("*", "")
    return re.sub(r"[^a-z0-9]", "", s.lower())


tests = [
    # heading,                      ingredient,                                 want
    ("10mM Fe quinate solution",    "10 mM Fe quinate solution* (see below)",   True),
    ("Phosphates solution",         "Phosphates solution",                       True),
    ("Phosphate buffer stock solution", "Phosphate buffer stock solution",       True),
    ("Solution A",                  "5\\% K$_2$HPO$_4$ solution (autoclaved)",   False),
]

for h, m, want in tests:
    got = base_name(h) == base_name(m)
    print(f"{'ok ' if got == want else 'FAIL'}  {got}  {h!r} vs {m!r}")

ok   True  '10mM Fe quinate solution' vs '10 mM Fe quinate solution* (see below)'
ok   True  'Phosphates solution' vs 'Phosphates solution'
ok   True  'Phosphate buffer stock solution' vs 'Phosphate buffer stock solution'
ok   False  'Solution A' vs '5\\% K$_2$HPO$_4$ solution (autoclaved)'


In [23]:
from collections import Counter

kinds = Counter()
unmatched = []

for _, row in clean.iterrows():
    tex = row["tex_text_clean"]
    monos = re.findall(r"\\mono\{([^}]*)\}", tex)
    keys = {base_name(m) for m in monos}
    external = {base_name(m) for m in monos if "see Medium No" in m}

    for h in re.findall(r"\\chu\{([^}]*:)\}", tex):
        h = h.strip().rstrip(":").strip()
        if not h:
            continue
        k = base_name(h)
        if not k:
            continue
        if k in external:
            kinds["external_ref"] += 1
        elif k in keys:
            kinds["sub_recipe"] += 1
        elif re.match(r"(?i)solution\s+[a-z0-9]+$", h):
            kinds["solution"] += 1
        else:
            kinds["unmatched"] += 1
            unmatched.append((row["grmd"], h))

print(kinds)
print()
for g, h in unmatched[:25]:
    print(f"  {g}  {h[:60]!r}")

Counter({'unmatched': 521, 'sub_recipe': 343, 'solution': 126, 'external_ref': 5})

  1436  'Mix components thoroughly, sparge the medium with a N$_2$-CO'
  1436  'Prior to use, add per liter the following solution (autoclav'
  1479  'Mix components thoroughly, adjust pH to 7.0 and autoclave un'
  1475  'Mix components thoroughly and autoclave under a N$_2$-CO$_2$'
  1468  'Mix components and autoclave under a N$_2$-CO$_2$ (80:20, v/'
  1462  'Mix components thoroughly and adjust pH to 6.5, then distrib'
  1458  'Mix components and autoclave. After cooling, aseptically and'
  1389  'Mix components, except NaHCO$_3$, and adjust pH 7.0. Bring t'
  1389  'Distribute the medium into culture vessels (e.g., 10 ml in H'
  1392  'Mix components thoroughly and adjust pH 7.5. Bring to a boil'
  1364  'Mix components, adjust pH to 8 and autoclave under a N$_2$ g'
  1364  'Readjust pH to 8, if necessary.  Separately autoclave and dr'
  1365  'Mix components thoroughly.  For preparation of solid me

In [24]:
def is_heading(text):
    """A heading is a short label ending in ':'. Instructions also end in ':'
    when they introduce a list ('add the following:'), so length separates
    them."""
    t = text.strip()
    return t.endswith(":") and len(t) < 60 and "." not in t[:-1]

In [25]:
from collections import Counter

kinds = Counter()
unmatched = []

for _, row in clean.iterrows():
    tex = row["tex_text_clean"]
    monos = re.findall(r"\\mono\{([^}]*)\}", tex)
    keys = {base_name(m) for m in monos}
    external = {base_name(m) for m in monos if "see Medium No" in m}

    for raw in re.findall(r"\\chu\{([^}]*)\}", tex):
        raw = raw.strip()
        if not raw:
            continue

        if not is_heading(raw):
            kinds["instruction"] += 1
            continue

        h = raw.rstrip(":").strip()
        k = base_name(h)
        if not k:
            continue

        if re.match(r"(?i)^(comment|note|\[note\])$", h):
            kinds["comment"] += 1
        elif k in external:
            kinds["external_ref"] += 1
        elif k in keys:
            kinds["sub_recipe"] += 1
        elif re.match(r"(?i)solution\s+[a-z0-9]+$", h):
            kinds["solution"] += 1
        else:
            kinds["unmatched"] += 1
            unmatched.append((row["grmd"], h))

print(kinds)
print(f"\nunmatched ({len(unmatched)}):")
for g, h in unmatched[:30]:
    print(f"  {g}  {h[:60]!r}")

Counter({'instruction': 1738, 'sub_recipe': 339, 'solution': 126, 'unmatched': 75, 'comment': 7, 'external_ref': 5})

unmatched (75):
  1365  'Selenite-tungstate solution'
  1398  'Neutralized sulfide solution'
  1398  'Vitamin solution CA'
  1387  'Na$_2$S·9H$_2$O solution'
  1386  'Bicarbonate solution'
  1386  'Na$_2$S·9H$_2$O solution'
  1347  "Modified Wolfe's Mineral solution"
  1342  'Ferrihydrite slurry'
  1286  'Sterilizationofsulfur'
  1287  'Sterilization of sulfur'
  1288  'Sterilization of sulfur'
  1284  'Sterilization of sulfur'
  1258  'Stearic solution'
  1232  'Concentrated Vibrio suspension'
  1209  '\\chu{Basal medium salts solution'
  1182  'Trace element solution'
  1130  'Vitamin solution II'
  1100  'CTM medium basis'
  1100  'Solution 1 (50 x stock)'
  1100  'Mixed solution'
  1100  'Vitamin solution A (1000 x stock)'
  1100  'Vitamin solution B (1000 x stock)'
  1139  'Casein solution'
  1107  'Trace elements solution'
  1081  'Soda based mineral medium'
  107

In [26]:
import difflib

for grmd, h in unmatched:
    tex = clean[clean["grmd"] == grmd]["tex_text_clean"].values[0]
    monos = [base_name(m) for m in re.findall(r"\\mono\{([^}]*)\}", tex)]
    near = difflib.get_close_matches(base_name(h), monos, n=1, cutoff=0.85)
    if near:
        print(f"{grmd}  {h[:40]!r}  ≈  {near[0][:40]!r}")

1365  'Selenite-tungstate solution'  ≈  'selenitetungustatesolution'
1347  "Modified Wolfe's Mineral solution"  ≈  'modifiedwolfessolution'
1258  'Stearic solution'  ≈  'stericsolution'
1182  'Trace element solution'  ≈  'traceelementssolution'
1130  'Vitamin solution II'  ≈  'vitaminsolutioni'
1107  'Trace elements solution'  ≈  'traceelementsolution'
971  "Modified Wolfe's minerals (10 x)"  ≈  'modifiedwolfesminerals10xseebelow'
820  'Trace mineral solution'  ≈  'tracemetalsolution'
628  "Modified Wolfe's mineral solution"  ≈  'modifiedwolfessolution'
576  'Trace metal solution SL12'  ≈  'traceelementsolutionsl12'
536  'Basal solution'  ≈  'basalsaltsolution'


In [27]:
for grmd in [1209, 995]:
    tex = clean[clean["grmd"] == grmd]["tex_text_clean"].values[0]
    for line in tex.splitlines():
        if "chu" in line and ("{" in line.replace("\\chu{", "", 1)):
            print(f"{grmd} → {line.strip()[:100]!r}")

1209 → '\\chu{\\chu{Basal medium salts solution:}'
995 → '\\chu{{Solution 1:}}'
995 → '\\chu{{Solution 2:}}'
995 → '\\chu{{Vitamin solution No. 6:}}'


In [28]:
RE_CHU_WRAPPER = re.compile(r"^\\chu\w*\s*\{?")

def heading_text(block):
    """Strip the \\chu wrapper and any doubled tag or brace.

    Source defects handled: `\\chu{\\chu{X:}` (tag typed twice, medium 1209)
    and `\\chu{{X:}}` (doubled braces around the whole heading, medium 995).
    """
    t = block.strip()
    while True:
        new = RE_CHU_WRAPPER.sub("", t).strip()
        if new == t:
            break
        t = new
    return t.strip("{}").strip()

In [29]:
for b in [r"\chu{\chu{Basal medium salts solution:}",
          r"\chu{{Solution 1:}}",
          r"\chu{Solution A:}",
          r"\chuSolutionB:}"]:
    print(f"{b!r:50} → {heading_text(b)!r}")

'\\chu{\\chu{Basal medium salts solution:}'        → 'Basal medium salts solution:'
'\\chu{{Solution 1:}}'                             → 'Solution 1:'
'\\chu{Solution A:}'                               → 'Solution A:'
'\\chuSolutionB:}'                                 → ':'


In [30]:
RE_BARE_TAG = re.compile(r"^\\chu(SolutionB|SolutionC|JCM)\b")

def heading_text(block):
    t = block.strip()
    m = RE_BARE_TAG.match(t)
    if m:
        return m.group(1), t[m.end():].strip().strip("{}").strip()
    ...

In [31]:
RE_BARE_TAG = re.compile(r"^\\chu(SolutionB|SolutionC|JCM)\b")
RE_CHU_WRAPPER = re.compile(r"^\\chu\w*\s*\{?")


def heading_text(block):
    r"""Strip the \chu wrapper from a block.

    Returns (tag, text). `tag` is the bare-tag name when the block uses one
    (\chuSolutionB, \chuSolutionC, \chuJCM), otherwise None — those carry
    their label in the tag itself rather than in the text.

    Source defects handled: `\chu{\chu{X:}` (tag typed twice, medium 1209)
    and `\chu{{X:}}` (doubled braces around the whole heading, medium 995).
    """
    t = block.strip()

    m = RE_BARE_TAG.match(t)
    if m:
        return m.group(1), t[m.end():].strip().strip("{}").strip()

    while True:
        new = RE_CHU_WRAPPER.sub("", t).strip()
        if new == t:
            break
        t = new

    return None, t.strip("{}").strip()

In [32]:
for b in [r"\chu{\chu{Basal medium salts solution:}",
          r"\chu{{Solution 1:}}",
          r"\chu{Solution A:}",
          r"\chuSolutionB:}",
          r"\chuJCM 32405 may grow better in a modified medium.}"]:
    print(f"{b[:45]!r:50} → {heading_text(b)}")

'\\chu{\\chu{Basal medium salts solution:}'        → (None, 'Basal medium salts solution:')
'\\chu{{Solution 1:}}'                             → (None, 'Solution 1:')
'\\chu{Solution A:}'                               → (None, 'Solution A:')
'\\chuSolutionB:}'                                 → ('SolutionB', ':')
'\\chuJCM 32405 may grow better in a modified m'   → ('JCM', '32405 may grow better in a modified medium.')
